# pynndescent

## Example 1: build an index and query it

In [ ]:
# Example 1: NNDescent finds the true nearest neighbours of well-separated points
import numpy as np
from pynndescent import NNDescent

rng = np.random.default_rng(0)
data = rng.normal(size=(500, 8)).astype(np.float32)
index = NNDescent(data, n_neighbors=10, random_state=0)
indices, distances = index.neighbor_graph
assert indices.shape == (500, 10) and (indices[:, 0] == np.arange(500)).all()  # each point is its own nearest
print("Example 1:", indices.shape, float(distances[:, 1].mean()))

## Example 2: recall against brute force

In [ ]:
# Example 2: the approximate neighbours agree with exact ones on almost every point
import numpy as np
from pynndescent import NNDescent
from sklearn.neighbors import NearestNeighbors

rng = np.random.default_rng(1)
data = rng.normal(size=(800, 10)).astype(np.float32)
approx, _ = NNDescent(data, n_neighbors=5, random_state=1).neighbor_graph
exact = NearestNeighbors(n_neighbors=5).fit(data).kneighbors(data, return_distance=False)
recall = np.mean([len(set(a) & set(e)) / 5 for a, e in zip(approx, exact)])
assert recall > 0.9, recall
print("Example 2: recall@5 =", round(float(recall), 3))

## Example 3: query new points

In [ ]:
# Example 3: prepare() then query() searches the index for points it has not seen
import numpy as np
from pynndescent import NNDescent

rng = np.random.default_rng(2)
data = rng.normal(size=(600, 6)).astype(np.float32)
index = NNDescent(data, n_neighbors=8, random_state=2)
index.prepare()
queries = data[:5] + 1e-4
found, dist = index.query(queries, k=3)
assert (found[:, 0] == np.arange(5)).all(), found
print("Example 3:", found.tolist())

## Example 4: other metrics

In [ ]:
# Example 4: cosine and manhattan distances are supported out of the box
import numpy as np
from pynndescent import NNDescent

rng = np.random.default_rng(3)
data = np.abs(rng.normal(size=(300, 5))).astype(np.float32)
for metric in ("cosine", "manhattan"):
    idx, dist = NNDescent(data, metric=metric, n_neighbors=4, random_state=3).neighbor_graph
    assert idx.shape == (300, 4) and (dist[:, 0] < 1e-5).all(), (metric, dist[:3])
print("Example 4: cosine and manhattan graphs built")

## Example 5: the scikit-learn transformer

In [ ]:
# Example 5: PyNNDescentTransformer returns a sparse k-neighbours graph like sklearn's KNeighborsTransformer
import numpy as np
from pynndescent import PyNNDescentTransformer

rng = np.random.default_rng(4)
data = rng.normal(size=(200, 4)).astype(np.float32)
graph = PyNNDescentTransformer(n_neighbors=6, random_state=4).fit_transform(data)
assert graph.shape == (200, 200) and graph.nnz >= 200 * 6, (graph.shape, graph.nnz)
print("Example 5:", type(graph).__name__, graph.shape, graph.nnz)